# What CMC does and how it maps onto Vib2Sound 
In Makishima's original setting, the "visual signal" is a lip-movement video, a noise-free, speaker-identifying cue. In our setting, the accelerometer plays exactly that role: it's body-contact signal that is unaffected by airborne noise from other birds, and it contains identity info the microphone alone can't reliably extract. 

Core idea: after training, the audio ecnode should produce representations that are *geometrically* close to the accelerometer representation when they belong to the same bird, and *orthogonal* to it when they don't. The loss enforces this via frame-wise cosine similarity. The AVC block that computes this is discarded after training (zero inference cost). 

| Makishima block | Vib2Sound equivalent                          
| --------        | --------                                      
| AudioBlock      | 8-layer CNN processing the mixture spectogram 
| VideoBlock      | accelerometer magnitude spectogram path (already a CNN)
| FusionBlock     | LSTM + FC layers that concatenate btoh modalities 
| AVCBlock        | New - a small projection head on top of separated output 

The **loss formula** from the paper (eq. 10) is: 
$$\mathcal{L}_{\text{CMC}} = \sum_{n} \sum_{j} \left[ \sum_{n' \neq n} d(c_{nj}^v, c_{n'j}^{\text{avc}}) - d(c_{nj}^v, c_{nj}^{\text{avc}}) \right]$$

where $d(a, b) = \frac{a^\top b}{\|a\| \|b\|}$ is cosine similarity, $c_{nj}^v$ is the accelerometer feature for bird $n$ at frame $j$, and $c_{nj}^{\text{avc}}$ is the AVC projection of the separated audio for bird $n$ at frame $j$. With $N = 2$ birds this reduces to one positive pair and one negative pair per frame.

The total loss is:

$$\mathcal{L} = \mathcal{L}_{\text{MSE}} + \lambda \mathcal{L}_{\text{CMC}}$$

Makishima found $\lambda = 1$ optimal — a reasonable starting point for you too, but worth sweeping over $(0.1, 1, 10)$.

# Implementation 
I'll add 2 things to the existing Vib2Sound code:
1. **AVCBlock** projection head
2. **CMCLoss** module 
everything else (CNN, LSTM, FC layers, MSE loss) stays untouched.

I need to be careful on these things before i run it:
1. **Where to tap the mask**: `AVCBlock` should operate on the LSTM/FC output *before* the sigmoid, so it sees the full real-valued feature space rather than the squashed $[0, 1]$ mask. I need to recheck if the model exposes this intermediate tensor (may need to split the `forward()` method to return it).
2. **Shared vs per-bird AVC weights**: Makishima shares AVC block weights across speakers. That works when all speakers are the same species (our case). I think it's fine if I start with shared weights since I can ablate separate weights later.
3. **Temporal alignment**: The accelerometer spectrogram may have a $\neq$ number of frequency bins and possibly a $\neq$ time resolution than the audio mask (our STFT uses FFT 384, hop 96, at $24414\text{ Hz}$). I need to make sure `AccelEmbedder`'s `AdaptiveAvgPool2d` and any upsampling/downsampling produce the same $T$ dimension as the mask before the cosine similarity is computed. Log both shapes on the first batch.